In [1]:
import pandas as pd

In [2]:
cleaned_Dataset = pd.read_csv('/content/clean_reddit_data.csv')

In [3]:
cleaned_Dataset.head()

,comment_id,post_id,subreddit,body,year,month,day,clean_text,timestamp
0,nu5zus1,1pjvhso,worldnews,> and overnight the narrative was accepted. N...,2025,12,15,overnight narrative accepted hardly anyone bel...,2025-12-15 15:10:46
1,m8dq55t,1i69kk9,worldnews,"U are away your taxes don’t pay for anything, ...",2025,1,21,away taxes pay anything makes bond fed buys mo...,2025-01-21 17:28:22
2,mg37xlt,1j2ychy,worldnews,That’s so incredibly stupid. During ww2 Europe...,2025,3,5,incredibly stupid divided different blocks sin...,2025-03-05 03:46:40
3,mjr6ukt,1jjfllw,worldnews,> The source said the Canadian Security Intell...,2025,3,26,source security intelligence service learned i...,2025-03-26 00:37:25
4,o48cox0,1qygesu,worldnews,"She is not conservative. She is normal, ration...",2026,2,8,conservative normal rational human cares state,2026-02-08 10:25:06


In [4]:
cleaned_Dataset.to_csv('clean_reddit_data.csv', index=False)
print("Saved!")

Saved!


In [5]:
import pandas as pd
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer

df = pd.read_csv('clean_reddit_data.csv')

# Remove empty rows — LDA will crash if it sees blank text
df = df.dropna(subset=['clean_text'])
df = df[df['clean_text'].str.strip() != '']

# Stopword removing

In [6]:

import spacy
import re
import pandas as pd
import nltk
nltk.download('stopwords', quiet=True)
from nltk.corpus import stopwords

nlp = spacy.load('en_core_web_sm')
nlp.max_length = 2000000

# 1. Tumhari dono files load karo
with open('/content/stopwords.txt', 'r', encoding='utf-8', errors='ignore') as f:
    file1_words = {line.strip().lower() for line in f if line.strip()}

with open('/content/trend_stopwords.txt', 'r', encoding='utf-8', errors='ignore') as f:
    file2_words = {line.strip().lower() for line in f if line.strip()}

# 2. Extra jo abhi bhi aa rahe the
extra_remove = {
    'not','hardly','actually','never','nothing','nobody','nowhere',
    'neither','nor','cannot','cant','wont','dont','isnt','arent',
    'wasnt','werent','havent','hasnt','hadnt','nope','yeah','yep',
    'say','said','says','saying','tell','told','ask','asked',
    'think','thought','thinking','know','knew','known',
    'conservative','normal','rational','currently','generally',
    'care','thing','things','stuff','way','lot','bit','kind',
    'human','person','people','man','men','woman','women','guy','guys',
    'taxis','datum','medium','corpus','erratum',
    'lol','lmao','omg','wtf','smh','imo','imho','reddit',
    'comment','post','edit','deleted','removed','upvote','downvote',
}

# 3. NLTK base
nltk_base = set(stopwords.words('english'))

# 4. Sab ek jagah merge
all_stopwords = nltk_base | file1_words | file2_words | extra_remove

# 5. Spacy mein inject
nlp.Defaults.stop_words.update(all_stopwords)
for word in all_stopwords:
    if word in nlp.vocab:
        nlp.vocab[word].is_stop = True

# 6. Important trend words protect karo
keep_these = {
    'nuclear','missile','invasion','sanction','tariff',
    'election','military','weapon','troop','soldier',
    'president','democracy','republican','democrat',
    'zelensky','trump','putin','nato','dictator',
    'poland','china','russia','ukraine','india','iran',
    'israel','taiwan','pakistan','japan','korea',
    'economy','trade','oil','energy','market','inflation',
    'border','territory','regime','protest','corruption',
    'court','office','national','security','power',
    'history','evidence','policy','propaganda','intelligence',
    'canadian','leadership','community','assessment','treaty',
    'alliance','occupation','ceasefire','airstrike','offensive',
    'casualties','referendum','sovereignty','annexation',
}
for word in keep_these:
    if word in nlp.vocab:
        nlp.vocab[word].is_stop = False

# 7. Spacy broken lemma fixes
lemma_fixes = {
    'taxes':'tax',  'taxis':'tax',
    'data':'data',  'datum':'data',
    'media':'media','medium':'media',
    'corps':'corps',
}

def process_batch(texts, batch_size=1000):
    results = []
    clean_texts = []

    for t in texts:
        if not isinstance(t, str):
            clean_texts.append('')
            continue
        t = t.lower()
        t = re.sub(r'http\S+|www\S+|https\S+', '', t)
        t = re.sub(r'\[.*?\]\(.*?\)', '', t)
        t = re.sub(r"n't|n't", '', t)
        t = re.sub(r"'s|'s", '', t)
        t = re.sub(r'[^a-z\s]', '', t)
        t = re.sub(r'\d{4}-\d{2}-\d{2}', '', t)
        t = re.sub(r'\b\w{1,2}\b', '', t)
        t = re.sub(r'\s+', ' ', t).strip()
        clean_texts.append(t)

    for doc in nlp.pipe(clean_texts, batch_size=batch_size,
                        disable=['ner','parser','senter']):
        tokens = []
        for token in doc:
            if token.is_stop or token.is_punct or token.is_space:
                continue
            lemma = lemma_fixes.get(token.text, token.lemma_)
            lemma = lemma_fixes.get(lemma, lemma)
            if lemma in all_stopwords:
                continue
            if len(lemma) < 3:
                continue
            tokens.append(lemma)

        results.append(" ".join(tokens) if tokens else None)

    return results

df['clean_text'] = process_batch(df['body'])
df = df[df['clean_text'].notna() & (df['clean_text'].str.strip() != '')].reset_index(drop=True)
# 1. pretty aur aise baaki fillers add karo
extra_remove = {
    'pretty','quite','rather','fairly','somewhat','slightly',
    'clearly','simply','obviously','literally','basically',
    'certainly','definitely','probably','possibly','perhaps',
    'mostly','mainly','largely','generally','typically',
    'country','countries',   # bahut generic hain trend ke liye
}

all_stopwords = all_stopwords | extra_remove

# Spacy mein update karo
for word in extra_remove:
    if word in nlp.vocab:
        nlp.vocab[word].is_stop = True

# 2. Row 4 jaisi choti rows drop karo — minimum 5 words hone chahiye
df['clean_text'] = process_batch(df['body'])

df = df[
    df['clean_text'].notna() &
    (df['clean_text'].str.strip() != '') &
    (df['clean_text'].str.split().str.len() >= 5)   # ← minimum 5 words
].reset_index(drop=True)

extra_remove = {
    # Generic verbs jo abhi bhi aa rahe hain
    'bring','brought','bringing',
    'hope','hoping','hoped',
    'option','options',
    'choose','chose','chosen',
    # Food words — worldnews trend mein useless
    'cheese','poutine','food','meal','eat','drink',
    'bread','meat','rice','fruit','vegetable',
    # Generic nouns
    'transaction','transact',
}

all_stopwords = all_stopwords | extra_remove
for word in extra_remove:
    if word in nlp.vocab:
        nlp.vocab[word].is_stop = True

# Dobara process karo
df['clean_text'] = process_batch(df['body'])

df = df[
    df['clean_text'].notna() &
    (df['clean_text'].str.strip() != '') &
    (df['clean_text'].str.split().str.len() >= 5)
].reset_index(drop=True)

print(f"✅ Done: {len(df)} rows")
print(df['clean_text'].head(10))  # 10 rows dekho ab


✅ Done: 95008 rows
0    overnight narrative accept exclusively maga cu...
1                  tax government bond fed money print
2    incredibly europe divide block alliance union ...
3    canadian security intelligence service indian ...
4    ukraine weapon european ukraine equipment desi...
5     courier deliver toy deliver blood donor hospital
6    trump road lead russia choice tie pattern inte...
7    demographic honest realise population concentr...
8                  voter afd represent vance couchfuer
9    ukraine bluesky novorossiysk ukrainian drone r...
Name: clean_text, dtype: object
✅ Done: 95008 rows
0    overnight narrative accept exclusively maga cu...
1                  tax government bond fed money print
2    incredibly europe divide block alliance union ...
3    canadian security intelligence service indian ...
4    ukraine weapon european ukraine equipment desi...
5     courier deliver toy deliver blood donor hospital
6    trump road lead russia choice tie pattern int

 CountVectorizer

In [7]:
cv = CountVectorizer(
    max_features=5000,
    max_df=0.85,       # ← NEW: removes words in 85%+ of comments
    min_df=10,         # ← NEW: removes very rare words
    stop_words='english'
)

cv_matrix = cv.fit_transform(df['clean_text'])
print("Done:", cv_matrix.shape)

Done: (95008, 5000)


TF-IDF

In [8]:
tfidf = TfidfVectorizer(
    max_features=5000,
    max_df=0.95,
    min_df=2,
    stop_words='english'
)

tfidf_matrix = tfidf.fit_transform(df['clean_text'])
print("Shape:", tfidf_matrix.shape)

Shape: (95008, 5000)


# LDA

In [9]:
from sklearn.decomposition import LatentDirichletAllocation

topic_counts = [10, 15, 20]
lda_models = {}

for n in topic_counts:
    print(f"Training LDA with {n} topics...")
    lda = LatentDirichletAllocation(
        n_components=n,      # number of topics to find
        random_state=42,     # fixes randomness so results are reproducible
        max_iter=10,         # how many times to optimize (more = slower but better)
        learning_method='online'  # processes data in batches, faster for large data
    )
    lda.fit(cv_matrix)
    lda_models[n] = lda

    # Perplexity = how "confused" the model is (lower = better fit)
    print(f"  Perplexity: {lda.perplexity(cv_matrix):.2f}")

Training LDA with 10 topics...
  Perplexity: 2640.86
Training LDA with 15 topics...
  Perplexity: 2927.76
Training LDA with 20 topics...
  Perplexity: 3257.16


In [10]:
!pip install gensim

In [11]:
from gensim.models.coherencemodel import CoherenceModel
from gensim.corpora import Dictionary

# Step 1 — Prepare text as list of word lists
# Gensim needs: [['word1','word2'], ['word3','word4'], ...]
# NOT a full sentence string
texts = [str(text).split() for text in df['clean_text']]

# Example of what texts looks like:
# [['climate', 'change', 'real'], ['war', 'ukraine', 'bad'], ...]

# Step 2 — Build Dictionary
# Maps every unique word to a number ID
# Like a personal phonebook: climate=1, war=2, troops=3 ...
dictionary = Dictionary(texts)

# Step 3 — Function to calculate coherence
def get_coherence(lda_model, vectorizer, texts, dictionary):

    vocab = vectorizer.get_feature_names_out()  # all 5000 words
    topics = []

    for topic in lda_model.components_:
        # Get top 10 words for this topic
        top_words = [vocab[i] for i in topic.argsort()[:-11:-1]]
        topics.append(top_words)

    # CoherenceModel checks how often top words appear
    # TOGETHER in the same real comments
    # c_v is the most reliable coherence method
    cm = CoherenceModel(
        topics=topics,
        texts=texts,
        dictionary=dictionary,
        coherence='c_v'
    )
    return cm.get_coherence()

# Step 4 — Check all 3 models
print("Calculating coherence scores...\n")
scores = {}

for n in topic_counts:
    score = get_coherence(lda_models[n], cv, texts, dictionary)
    scores[n] = score
    print(f"Topics: {n}  →  Coherence Score: {score:.4f}")

# Step 5 — Pick the best
best_n = max(scores, key=scores.get)
print(f"\n✅ Best number of topics: {best_n}")
best_lda = lda_models[best_n]

Calculating coherence scores...

Topics: 10  →  Coherence Score: 0.6141
Topics: 15  →  Coherence Score: 0.5824
Topics: 20  →  Coherence Score: 0.5685

✅ Best number of topics: 10


Perplexity said  → 10 topics ✅
Coherence said   → 10 topics ✅

No confusion — 10 topics is clearly the winner

In [12]:
import pickle

with open('lda_model.pkl', 'wb') as f:
    pickle.dump(lda_models[10], f)

print("Model saved as lda_model.pkl")

Model saved as lda_model.pkl


In [13]:
from wordcloud import WordCloud
import matplotlib.pyplot as plt
import os

os.makedirs('outputs', exist_ok=True)

vocab = cv.get_feature_names_out()

for i, topic in enumerate(lda_models[10].components_):
    word_weights = {vocab[j]: topic[j] for j in topic.argsort()[:-51:-1]}

    wc = WordCloud(width=800, height=400, background_color='white')
    wc.generate_from_frequencies(word_weights)

    plt.figure(figsize=(10, 5))
    plt.imshow(wc)
    plt.axis('off')
    plt.title(f'Topic {i}', fontsize=18, fontweight='bold')
    plt.savefig(f'outputs/topic_{i}_wordcloud.png', dpi=150)
    plt.close()
    print(f"Saved topic_{i}_wordcloud.png ✅")

print("\nAll word clouds done!")

Saved topic_0_wordcloud.png ✅
Saved topic_1_wordcloud.png ✅
Saved topic_2_wordcloud.png ✅
Saved topic_3_wordcloud.png ✅
Saved topic_4_wordcloud.png ✅
Saved topic_5_wordcloud.png ✅
Saved topic_6_wordcloud.png ✅
Saved topic_7_wordcloud.png ✅
Saved topic_8_wordcloud.png ✅
Saved topic_9_wordcloud.png ✅

All word clouds done!
